# Explore raw market data

Scratch notebook for poking at the raw parquet streams (BBO `front_levels`, `trade`, funding `fri`)
straight off disk. Reads via `boba.io` loaders only — **no dataset/feature classes**, so what you
see here is exactly what the ETL ingests.

Data is organised in ~24h **blocks** (`holocron.{ts}.{idx}`) per **listing** token
(e.g. `bin_doge_usdt_p`). The data root comes from `data_dir` in `settings.local.toml`.

In [ ]:
import polars as pl

from boba.io import DATA_DIR, list_blocks, load_block

pl.Config.set_tbl_rows(12)
print("DATA_DIR:", DATA_DIR)

## What's on disk

Each file is `holocron.{ts}.{idx}.{listing}.{data_type}.parquet`. Enumerate the available
listing tokens, then the blocks present for one of them.

In [ ]:
listings = sorted({".".join(f.name.split(".")[3:-2])
                   for f in DATA_DIR.glob("holocron.*.front_levels.parquet")})
print(f"{len(listings)} listings:")
print("  " + "\n  ".join(listings))

In [ ]:
LISTING = "bin_doge_usdt_p"

blocks = list_blocks(LISTING, "front_levels")
print(f"{len(blocks)} blocks for {LISTING}; using the first one")
BLOCK = blocks[0]
BLOCK

## BBO (`front_levels`)

Best bid/ask snapshots. `rx_time` is our receive clock (ns, UTC); `exchange_time` is the venue
clock — the gap is feed latency. Prices/qtys are float64.

In [ ]:
fl = load_block(BLOCK, LISTING, "front_levels").sort("rx_time")
print(fl.shape)
fl.head()

In [ ]:
# Mid + spread, downsampled to ~1s so it's plottable without choking on millions of rows.
mid = (
    fl.select(
        "rx_time",
        ((pl.col("bid_prc") + pl.col("ask_prc")) / 2).alias("mid"),
        (pl.col("ask_prc") - pl.col("bid_prc")).alias("spread"),
    )
    .group_by_dynamic("rx_time", every="1s")
    .agg(pl.col("mid").last(), pl.col("spread").mean())
)
mid.head()

## Trades (`trade`)

`aggressor` is `Bid`/`Ask` (taker side); the dataset maps `Bid → +1`, `Ask → -1`.

In [ ]:
td = load_block(BLOCK, LISTING, "trade").sort("rx_time")
print(td.shape)
td.select("rx_time", "exchange_time", "aggressor", "prc", "qty").head()

In [ ]:
# Signed volume by taker side.
td.group_by("aggressor").agg(
    pl.len().alias("n_trades"),
    pl.col("qty").sum().alias("total_qty"),
)

## Funding (`fri`)

Indicative funding rate + next funding time (perp listings only).

In [ ]:
fri = load_block(BLOCK, LISTING, "fri").sort("rx_time")
print(fri.shape)
fri.select("rx_time", "indicative_funding_rate", "next_funding_time").head()

## Quick plot

Requires `matplotlib` in the env (see `notebooks/README.md`).

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(mid["rx_time"], mid["mid"], lw=0.7)
ax1.set_ylabel("mid")
ax1.set_title(f"{LISTING}  —  {BLOCK}")
ax2.plot(mid["rx_time"], mid["spread"], lw=0.7, color="C1")
ax2.set_ylabel("mean spread")
fig.tight_layout()